# eICU preprocessing notebook

Prepare hourly labs and vitals, then build task datasets.

Prereqs:
- Run this notebook from the eICU CSV root (patient.csv, lab.csv, vitalPeriodic.csv, etc.).
- Place the FIDDLE population CSVs in population/ (see link below).
- Create figs/ before saving plots; preprocessed/ is created when needed.

In [ ]:
%load_ext autoreload
%autoreload 2
# activates line execution
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

Place this notebook in the eICU root directory where the raw CSV files are located.

Download the population/ folder (patient IDs and outcomes) from FIDDLE:
- Source: https://github.com/MLD3/FIDDLE
- Download link (request access if needed): https://drive.google.com/drive/folders/1RM-igV1OWk3ZyEu5FC1XnG9qTzJii8T9?usp=drive_link

In [ ]:
import json
import os
import pickle
import random

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(override=True)

random.seed(42)

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

## Quick checks on raw tables

In [ ]:
df_nurse = pd.read_csv('nurseCharting.csv')
df_nurse.shape
df_nurse.head(5)

In [ ]:
df_pe = pd.read_csv('physicalExam.csv', nrows=100000)
df_pe.shape
df_pe.head(5)

In [ ]:
df_pe['physicalexampath'].value_counts()

# find GCS

df_pe[df_pe['physicalexampath'].str.contains('GCS', na=False)]['physicalexampath'].unique()

In [ ]:
df_patients = pd.read_csv("patient.csv")

# replace age = > 89 with 90
df_patients["age"] = (
    df_patients["age"].replace("> 89", 90).fillna(55).astype(int)
)
df_patients["age"].value_counts()

## Labs: select frequent tests and aggregate hourly

In [ ]:
# 39M rows
df_lab = pd.read_csv("lab.csv", nrows=10_000_000)
df_lab = df_lab.sort_values(by=["patientunitstayid", "labresultoffset"])
df_lab.shape

df_lab.head()

In [ ]:
dff = df_lab[["labname", "labmeasurenamesystem"]].drop_duplicates()

len(lab_cols)
dff = dff[dff["labname"].isin(lab_cols)]

dff.to_csv("preprocessed/lab_units.csv", index=False)

# convert to dictionary labname: labunit
lab_units = {}
for _, row in dff.iterrows():
    lab_units[row["labname"]] = row["labmeasurenamesystem"]
lab_units

In [ ]:
df_lab.shape
df_lab_counts = df_lab["labname"].value_counts()
df_lab_counts = df_lab_counts / len(df_lab["patientunitstayid"].unique())
df_lab_counts = df_lab_counts[df_lab_counts > 0.1]
df_lab_counts

lab_cols = df_lab_counts.index.tolist()
lab_cols

# bar plot
fig = px.bar(df_lab_counts, x=df_lab_counts.index, y=df_lab_counts.values)
_ = fig.update_layout(
    title="Lab tests prevalence",
    xaxis_title="Lab test",
    yaxis_title="Prevalence",
    xaxis_tickangle=-45,
)
fig.show()

# save png figs/frequent_lab_tests.png
# fig.write_image("figs/frequent_lab_tests.png", width=800, height=400)

# write to html
fig.write_html("figs/frequent_lab_tests.html", include_plotlyjs="cdn")

In [ ]:
df_lab.head(40)
df_lab.shape

df_lab2 = df_lab[df_lab["labname"].isin(lab_cols)]
df_lab2.shape

df_lab2_pivot = df_lab2[
    ["patientunitstayid", "labresultoffset", "labname", "labresult"]
].pivot_table(
    index=["patientunitstayid", "labresultoffset"],
    columns="labname",
    values="labresult",
    aggfunc="last",
).reset_index()

# df_lab["labresulttext"].value_counts().tolist()

In [ ]:
# Floor labresultoffset to 1_hour windows
df_lab2_pivot["offset_1hour"] = (df_lab2_pivot["labresultoffset"] // 60) + 1
# df_lab2_pivot.head(50)

# Group by patient and time window
grouped = df_lab2_pivot.groupby(["patientunitstayid", "offset_1hour"])

# For each group, take the last non-null value per column
# df_lab2_pivot_agg = grouped[lab_cols].agg(
#     lambda x: x.dropna().iloc[-1] if not x.dropna().empty else pd.NA
# ).reset_index().dropna(subset=lab_cols, how="all")

# Then group and take the last non-null values per group
df_lab2_pivot_agg = (
    df_lab2_pivot.groupby(["patientunitstayid", "offset_1hour"], sort=False)[lab_cols]
    .last()
    .reset_index()
    .dropna(subset=lab_cols, how="all")
)

print("Shape before aggregation:", df_lab2_pivot.shape)
print("Shape after aggregation:", df_lab2_pivot_agg.shape)

In [ ]:
df_lab2_pivot_agg.fillna(0).sum().sum()

In [ ]:
df_lab2_pivot_agg

# save to csv
df_lab2_pivot_agg.to_csv(f"preprocessed/lab_agg.csv", index=False)

## Vitals: aggregate hourly

In [ ]:
# 146M rows
vitalp_cols = [
    "temperature",
    "sao2",
    "heartrate",
    "respiration",
    "cvp",
    "etco2",
    "systemicsystolic",
    "systemicdiastolic",
    "systemicmean",
    "pasystolic",
    "padiastolic",
    "pamean",
]
df_vitalp = (
    pd.read_csv("vitalPeriodic.csv", nrows=10_000_000)
    .sort_values(by=["patientunitstayid", "observationoffset"])
    .dropna(subset=vitalp_cols, how="all")
)
df_vitalp.shape

print("Number of patients:", df_vitalp["patientunitstayid"].nunique())

# df_vitalp.head(50)
# df_vitalp.columns

# Floor observationoffset to 1_hour windows
df_vitalp["offset_1hour"] = (df_vitalp["observationoffset"] // 60) + 1
# df_vitalp.head(50)

# Group by patient and time window
grouped = df_vitalp.groupby(["patientunitstayid", "offset_1hour"])

print("starting aggregation...")
# For each group, take the last non-null value per column
# df_vitalp_agg = grouped[vitalp_cols].agg(
#     lambda x: x.dropna().iloc[-1] if not x.dropna().empty else pd.NA
# ).reset_index().dropna(subset=vitalp_cols, how="all")

# Then group and apply .last()
df_vitalp_agg = (
    df_vitalp.groupby(["patientunitstayid", "offset_1hour"], sort=False)[vitalp_cols]
    .last()
    .reset_index()
    .dropna(subset=vitalp_cols, how="all")
)

print("Shape before aggregation:", df_vitalp.shape)
print("Shape after aggregation:", df_vitalp_agg.shape)

# missingrate
# df_vitalp_agg.head(50)
df_vitalp[vitalp_cols].isnull().sum() / df_vitalp.shape[0] * 100

df_vitalp_agg[vitalp_cols].isnull().sum() / df_vitalp_agg.shape[0] * 100

# save to csv
df_vitalp_agg.to_csv("preprocessed/vitalPeriodic_agg.csv", index=False)

In [ ]:
df_vitalp_agg.fillna(0).sum().sum()

In [ ]:
df_vitalp_agg.fillna(0).sum().sum()

## Feature mapping and summaries

In [ ]:
lab_cols = [
    "bedside glucose",
    "potassium",
    "sodium",
    "glucose",
    "Hgb",
    "chloride",
    "creatinine",
    "Hct",
    "BUN",
    "calcium",
    "bicarbonate",
    "platelets x 1000",
    "WBC x 1000",
    "RBC",
    "MCV",
    "MCHC",
    "MCH",
    "RDW",
    "anion gap",
    "MPV",
    "magnesium",
    "-lymphs",
    "-monos",
    "-eos",
    "-polys",
    "-basos",
    "paO2",
    "paCO2",
    "pH",
    "albumin",
    "phosphate",
    "HCO3",
    "FiO2",
    "AST (SGOT)",
    "ALT (SGPT)",
    "total protein",
    "alkaline phos.",
    "total bilirubin",
    "O2 Sat (%)",
    "Base Excess",
    "PT - INR",
    "PT",
    "PTT",
    "lactate",
    "Total CO2",
    "ionized calcium",
    "troponin - I",
    "PEEP",
    "Temperature",
    "TV",
    "Methemoglobin",
    "Carboxyhemoglobin",
    "urinary specific gravity",
    "CPK",
    "-bands",
    "direct bilirubin",
    "Base Deficit",
    "O2 Content",
    "Vent Rate",
    "Respiratory Rate",
    "CPK-MB",
    "triglycerides",
    "Vancomycin - trough",
    "Oxyhemoglobin",
    "BNP",
    "fibrinogen",
    "TSH",
    "lipase",
    "troponin - T",
    "serum osmolality",
    "ammonia",
    "Pressure Support",
    "LPM O2",
    "Vancomycin - random",
    "LDH",
    "total cholesterol",
    "prealbumin",
    "WBC's in urine",
    "CPK-MB INDEX",
    "HDL",
    "urinary sodium",
    "Fe",
    "LDL",
    "amylase",
    "urinary creatinine",
    "uric acid",
    "Ferritin",
    "TIBC",
    "CRP",
    "Vitamin B12",
]

vitalp_cols = [
    "temperature",
    "sao2",
    "heartrate",
    "respiration",
    "cvp",
    "etco2",
    "systemicsystolic",
    "systemicdiastolic",
    "systemicmean",
    "pasystolic",
    "padiastolic",
    "pamean",
]

In [ ]:
df_labs = pd.read_csv("preprocessed/lab_agg.csv")
df_vitals = pd.read_csv("preprocessed/vitalPeriodic_agg.csv")

df_patients = pd.read_csv("patient.csv")

# replace age = > 89 with 90
df_patients["age"] = (
    df_patients["age"].replace("> 89", 90).fillna(55).astype(int)
)
df_patients["age"].value_counts()

In [ ]:
df_diag = pd.read_csv("diagnosis.csv")
df_diag.shape

In [ ]:
lab_cols = pd.read_csv("preprocessed/lab_units.csv")

lab_cols

In [ ]:
lab_summary={
 'PT - INR': {'unit': 'ratio', 'new_name': 'INR'},
 'PT': {'unit': 'sec', 'new_name': 'Prothrombin_Time'},
 'PTT': {'unit': 'sec', 'new_name': 'Partial_Thromboplastin_Time'},
 'BUN': {'unit': 'mg/dL', 'new_name': 'Blood_Urea_Nitrogen'},
 '-lymphs': {'unit': '%', 'new_name': 'Lymphocytes_Percent'},
 'chloride': {'unit': 'mmol/L', 'new_name': 'Chloride'},
 'RBC': {'unit': 'M/mcL', 'new_name': 'Red_Blood_Cells'},
 'bicarbonate': {'unit': 'mmol/L', 'new_name': 'Bicarbonate'},
 'calcium': {'unit': 'mg/dL', 'new_name': 'Calcium_Total'},
 'glucose': {'unit': 'mg/dL', 'new_name': 'Glucose_Serum'},
 'AST (SGOT)': {'unit': 'Units/L', 'new_name': 'AST'},
 'ALT (SGPT)': {'unit': 'Units/L', 'new_name': 'ALT'},
 'anion gap': {'unit': float('nan'), 'new_name': 'Anion_Gap'},
 '-polys': {'unit': '%', 'new_name': 'Neutrophils_Percent'},
 'total bilirubin': {'unit': 'mg/dL', 'new_name': 'Total_Bilirubin'},
 'alkaline phos.': {'unit': 'Units/L', 'new_name': 'Alkaline_Phosphatase'},
 'RDW': {'unit': '%', 'new_name': 'Red_Cell_Distribution_Width'},
 'creatinine': {'unit': 'mg/dL', 'new_name': 'Creatinine'},
 'Hgb': {'unit': 'g/dL', 'new_name': 'Hemoglobin'},
 'potassium': {'unit': 'mmol/L', 'new_name': 'Potassium'},
 'total protein': {'unit': 'g/dL', 'new_name': 'Total_Protein'},
 '-eos': {'unit': '%', 'new_name': 'Eosinophils_Percent'},
 'albumin': {'unit': 'g/dL', 'new_name': 'Albumin'},
 'WBC x 1000': {'unit': 'K/mcL', 'new_name': 'White_Blood_Cells'},
 'sodium': {'unit': 'mmol/L', 'new_name': 'Sodium'},
 'MCV': {'unit': 'fL', 'new_name': 'Mean_Corpuscular_Volume'},
 'Hct': {'unit': '%', 'new_name': 'Hematocrit'},
 'MCH': {'unit': 'pg', 'new_name': 'Mean_Corpuscular_Hemoglobin'},
 '-basos': {'unit': '%', 'new_name': 'Basophils_Percent'},
 'MCHC': {'unit': 'g/dL', 'new_name': 'Mean_Corpuscular_Hemoglobin_Concentration'},
 '-monos': {'unit': '%', 'new_name': 'Monocytes_Percent'},
 'platelets x 1000': {'unit': 'K/mcL', 'new_name': 'Platelets'},
 'HCO3': {'unit': 'mmol/L', 'new_name': 'Bicarbonate_Alt'},
 'pH': {'unit': float('nan'), 'new_name': 'Blood_pH'},
 'FiO2': {'unit': '%', 'new_name': 'Fraction_Inspired_Oxygen'},
 'paCO2': {'unit': 'mm Hg', 'new_name': 'Arterial_CO2_Pressure'},
 'paO2': {'unit': 'mm Hg', 'new_name': 'Arterial_O2_Pressure'},
 'Base Deficit': {'unit': 'mEq/L', 'new_name': 'Base_Deficit'},
 'LPM O2': {'unit': 'L/min', 'new_name': 'Oxygen_Flow_Rate'},
 'fibrinogen': {'unit': 'mg/dL', 'new_name': 'Fibrinogen'},
 'magnesium': {'unit': 'mg/dL', 'new_name': 'Magnesium'},
 'lactate': {'unit': 'mmol/L', 'new_name': 'Lactate'},
 'CPK': {'unit': 'Units/L', 'new_name': 'Creatine_Phosphokinase'},
 'BNP': {'unit': 'pg/mL', 'new_name': 'B_Natriuretic_Peptide'},
 'troponin - I': {'unit': 'ng/mL', 'new_name': 'Troponin_I'},
 'CPK-MB': {'unit': 'ng/mL', 'new_name': 'CPK_MB'},
 'phosphate': {'unit': 'mg/dL', 'new_name': 'Phosphate'},
 'direct bilirubin': {'unit': 'mg/dL', 'new_name': 'Direct_Bilirubin'},
 'Total CO2': {'unit': float('nan'), 'new_name': 'Total_CO2'},
 "WBC's in urine": {'unit': float('nan'), 'new_name': 'Urine_WBCs'},
 'urinary specific gravity': {'unit': float('nan'), 'new_name': 'Urine_Specific_Gravity'},
 'bedside glucose': {'unit': 'mg/dL', 'new_name': 'Glucose_Bedside'},
 'Base Excess': {'unit': 'mEq/L', 'new_name': 'Base_Excess'},
 'TSH': {'unit': 'mcU/ml', 'new_name': 'Thyroid_Stimulating_Hormone'},
 'ammonia': {'unit': 'mcg/dL', 'new_name': 'Ammonia'},
 '-bands': {'unit': '%', 'new_name': 'Band_Neutrophils_Percent'},
 'LDL': {'unit': 'mg/dL', 'new_name': 'LDL_Cholesterol'},
 'total cholesterol': {'unit': 'mg/dL', 'new_name': 'Total_Cholesterol'},
 'HDL': {'unit': 'mg/dL', 'new_name': 'HDL_Cholesterol'},
 'triglycerides': {'unit': 'mg/dL', 'new_name': 'Triglycerides'},
 'O2 Sat (%)': {'unit': '%', 'new_name': 'Oxygen_Saturation'},
 'Methemoglobin': {'unit': '%', 'new_name': 'Methemoglobin'},
 'Carboxyhemoglobin': {'unit': '%', 'new_name': 'Carboxyhemoglobin'},
 'ionized calcium': {'unit': 'mg/dL', 'new_name': 'Calcium_Ionized'},
 'prealbumin': {'unit': 'mg/dL', 'new_name': 'Prealbumin'},
 'Vancomycin - trough': {'unit': 'mcg/mL', 'new_name': 'Vancomycin_Trough'},
 'Fe': {'unit': 'mcg/dL', 'new_name': 'Iron_Serum'},
 'lipase': {'unit': 'Units/L', 'new_name': 'Lipase'},
 'LDH': {'unit': 'Units/L', 'new_name': 'Lactate_Dehydrogenase'},
 'Vancomycin - random': {'unit': 'mcg/mL', 'new_name': 'Vancomycin_Random'},
 'Respiratory Rate': {'unit': '/min', 'new_name': 'Resp_Rate'},
 'serum osmolality': {'unit': 'mOsm/kg H2O', 'new_name': 'Serum_Osmolality'},
 'Vent Rate': {'unit': '/min', 'new_name': 'Ventilator_Rate'},
 'PEEP': {'unit': 'cm H2O', 'new_name': 'Positive_End_Expiratory_Pressure'},
 'CPK-MB INDEX': {'unit': '%', 'new_name': 'CPK_MB_Index'},
 'MPV': {'unit': 'fL', 'new_name': 'Mean_Platelet_Volume'},
 'Temperature': {'unit': '&#176;C', 'new_name': 'Body_Temperature_C'},
 'TV': {'unit': 'mls', 'new_name': 'Tidal_Volume'},
 'troponin - T': {'unit': 'ng/mL', 'new_name': 'Troponin_T'},
 'Pressure Support': {'unit': 'cm H2O', 'new_name': 'Ventilator_Pressure_Support'},
 'Oxyhemoglobin': {'unit': '%', 'new_name': 'Oxyhemoglobin'},
 'O2 Content': {'unit': 'mls/dL', 'new_name': 'Oxygen_Content'}
}


len(lab_summary)

In [ ]:
for _, row in tqdm(lab_cols.iterrows(), total=lab_cols.shape[0]):
    unit = row["labmeasurenamesystem"]

    data = df_labs[row["labname"]]
    overall_missing_rate = data.isnull().sum() / len(data) * 100
    avg_count_per_patient = (
        df_labs.groupby("patientunitstayid")[row["labname"]].count().mean()
    )

    # number of possible values
    n_possible_values = len(data.unique())

    # compute min, max, mean, std, q0.025, q0.975, q0.25, median, q0.75
    min_val = data.min()
    max_val = data.max()
    mean_val = data.mean()
    std_val = data.std()
    q0_025 = data.quantile(0.025)
    q0_975 = data.quantile(0.975)
    q0_25 = data.quantile(0.25)
    median_val = data.median()
    q0_75 = data.quantile(0.75)

    lab_summary[row["labname"]].update(
        {
            # "unit": unit,
            "overall_missing_rate": overall_missing_rate,
            "avg_count_per_patient": avg_count_per_patient,
            "n_possible_values": n_possible_values,
            "mean": mean_val,
            "std": std_val,
            "min": min_val,
            "q0.025": q0_025,
            "q0.25": q0_25,
            "median": median_val,
            "q0.75": q0_75,
            "q0.975": q0_975,
            "max": max_val,
        }
    )

# save to json
with open("preprocessed/lab_summary.json", "w") as f:
    json.dump(lab_summary, f, indent=4)

In [ ]:
vital_summary = {
    "temperature": {"unit": "°C", "new_name": "Body_Temperature_C"},
    "sao2": {"unit": "%", "new_name": "Arterial_Oxygen_Saturation"},
    "heartrate": {"unit": "bpm", "new_name": "Heart_Rate"},
    "respiration": {"unit": "breaths/min", "new_name": "Respiratory_Rate"},
    "cvp": {"unit": "mm Hg", "new_name": "Central_Venous_Pressure"},
    "etco2": {"unit": "mm Hg", "new_name": "End_Tidal_CO2"},
    "systemicsystolic": {"unit": "mm Hg", "new_name": "Systolic_BP_NIBP"},
    "systemicdiastolic": {"unit": "mm Hg", "new_name": "Diastolic_BP_NIBP"},
    "systemicmean": {"unit": "mm Hg", "new_name": "Mean_Arterial_Pressure_NIBP"},
    "pasystolic": {"unit": "mm Hg", "new_name": "Systolic_BP_Arterial"},
    "padiastolic": {"unit": "mm Hg", "new_name": "Diastolic_BP_Arterial"},
    "pamean": {"unit": "mm Hg", "new_name": "Mean_Arterial_Pressure_Arterial"},
}

for col in vitalp_cols:
    data = df_vitals[col]
    overall_missing_rate = data.isnull().sum() / len(data) * 100
    avg_count_per_patient = df_vitals.groupby("patientunitstayid")[col].count().mean()

    # number of possible values
    n_possible_values = len(data.unique())

    # compute min, max, mean, std, q0.025, q0.975, q0.25, median, q0.75
    min_val = data.min()
    max_val = data.max()
    mean_val = data.mean()
    std_val = data.std()
    q0_025 = data.quantile(0.025)
    q0_975 = data.quantile(0.975)
    q0_25 = data.quantile(0.25)
    median_val = data.median()
    q0_75 = data.quantile(0.75)

    vital_summary[col].update(
        {
            "overall_missing_rate": overall_missing_rate,
            "avg_count_per_patient": avg_count_per_patient,
            "n_possible_values": n_possible_values,
            "mean": mean_val,
            "std": std_val,
            "min": min_val,
            "q0.025": q0_025,
            "q0.25": q0_25,
            "median": median_val,
            "q0.75": q0_75,
            "q0.975": q0_975,
            "max": max_val,
        }
    )

# save to json
with open("preprocessed/vital_summary.json", "w") as f:
    json.dump(vital_summary, f, indent=4)

In [ ]:
# load lab_summary.json
with open("preprocessed/lab_summary.json", "r") as f:
    lab_summary = json.load(f)

# load vital_summary.json
with open("preprocessed/vital_summary.json", "r") as f:
    vital_summary = json.load(f)

units = {}

for _, value in lab_summary.items():
    units[value["new_name"]] = {
        "unit": value["unit"],
        "mean": value["mean"],
        "std": value["std"],
        "quantile2.5": value["q0.025"],
        "quantile97.5": value["q0.975"],
    }

for _, value in vital_summary.items():
    units[value["new_name"]] = {
        "unit": value["unit"],
        "mean": value["mean"],
        "std": value["std"],
        "quantile2.5": value["q0.025"],
        "quantile97.5": value["q0.975"],
    }

# save to units.json
with open("preprocessed/units.json", "w") as f:
    json.dump(units, f, indent=4)

In [ ]:
dict_map_names = {k: v["new_name"] for k, v in lab_summary.items()}
df_labs.rename(columns=dict_map_names, inplace=True)

dict_map_names = {k: v["new_name"] for k, v in vital_summary.items()}
df_vitals.rename(columns=dict_map_names, inplace=True)

## Population labels and task setup

In [ ]:
xx = pd.read_csv("population/ARF_4h.csv")

xx.head()
xx.shape

(xx["ARF_LABEL"] == 1).sum(), (xx["ARF_LABEL"] == 0).sum()

xx["ARF_ONSET_TIME"].describe()
# show histogram
fig = px.histogram(xx, x="ARF_ONSET_TIME", nbins=100)
_ = fig.update_layout(
    title="Shock ONSET TIME",
    xaxis_title="Shock ONSET TIME",
    yaxis_title="Count",
)
fig.show()

ARF_48_th = 48 * 60

# drop rows with ARF_ONSET_TIME < 48h. also keep nans
xx = xx[(xx["ARF_ONSET_TIME"] >= ARF_48_th) | (xx["ARF_ONSET_TIME"].isna())]

xx.shape
(xx["ARF_LABEL"] == 1).sum(), (xx["ARF_LABEL"] == 0).sum()

# save to csv
xx.to_csv("population/ARF_48h.csv", index=False)

In [ ]:
TASKS = [
    "mortality_48h",
    "Shock_4h",
    "Shock_12h",
    "ARF_4h",
    "ARF_12h",
    "Shock_48h",
    "ARF_48h",
    "LUNG_24h",
]

TASK = TASKS[-2]

# extract history from the task name
HISTORY = int(TASK.split("_")[1][:-1])

labels = pd.read_csv(f"population/{TASK}.csv")
LABEL = labels.columns[-1]

TASK, LABEL, HISTORY

list_ids = labels.ID.unique().tolist()[:]

# number of subjects
print(f"Number of subjects: {len(labels.ID.unique())}")

# prevlance of the label
print(f"Prevalence of the label: {labels[LABEL].mean() * 100:.2f}%")

In [ ]:
df_diags = pd.read_csv("diagnosis.csv")
df_diags.shape

In [ ]:
pid=141432
df_diags[df_diags['patientunitstayid'] == pid]

## Build per-patient time series and save train/test

In [ ]:
def process_func(pid):
    # create ts dataframe
    df_lab = df_labs[df_labs["patientunitstayid"] == pid]
    df_diag = df_diags[df_diags["patientunitstayid"] == pid]
    print(pid)
    term

    df_vital = df_vitals[df_vitals["patientunitstayid"] == pid]

    # merge based on patientunitstayid and offset_1hour
    df_ts = (
        pd.merge(
            df_lab,
            df_vital,
            on=["patientunitstayid", "offset_1hour"],
            how="outer",
        )
        .rename(columns={"offset_1hour": "Hours"})
    )

    if (df_ts["Hours"].max() < HISTORY / 2) or (len(df_ts) < 1):
        # if the max time is less than 12h, we don't need to keep this patient
        # print(df_ts["Hours"].max(),"**\n")
        return None

    # if (df_ts["Hours"].max() < HISTORY/2) or (len(df_ts) < 1):
    #     # if the max time is less than 12h, we don't need to keep this patient
    #     # print(df_ts["Hours"].max(),'**\n')
    #     return "E1"

    # if (len(df_ts) < 1):
    #     # if the max time is less than 12h, we don't need to keep this patient
    #     # print(df_ts["Hours"].max(),'**\n')
    #     return "E2"
    df_patients

    age = df_patients[df_patients["patientunitstayid"] == pid]["age"].values[0]
    gender = df_patients[df_patients["patientunitstayid"] == pid]["gender"].values[0]
    gender = 1 if gender == "Male" else 0

    data_patient = {
        "sid": pid,
        "hadm_id": [0],
        "covariates": [[age, gender]],
        "codes": None,
        "ts": [df_ts],
        f"label_{TASK}": [labels[labels["ID"] == pid][LABEL].values[0]],
    }

    return data_patient

In [ ]:
import multiprocessing as mp
from functools import partial

N_PROCESSES = mp.cpu_count()  # Use all available CPU cores
# N_PROCESSES = 1
partial_func = partial(process_func)

with mp.Pool(N_PROCESSES) as pool:
    all_results = list(
        tqdm(pool.imap(partial_func, list_ids), total=len(list_ids), leave=True)
    )

# remove None values
all_results = [x for x in all_results if x is not None]

# print the number of patients
print(f"Number of patients: {len(all_results)}")

In [ ]:
# split to train, test 80/20
from sklearn.model_selection import train_test_split

train, test = train_test_split(all_results, test_size=0.2, random_state=42)
print(f"Number of train samples: {len(train)}")
print(f"Number of test samples: {len(test)}")

# save to pickle
# create a new folder preprocessed/TASK
os.makedirs(f"preprocessed/{TASK}", exist_ok=True)
with open(f"preprocessed/{TASK}/trainDataset.pkl", "wb") as f:
    pickle.dump(train, f)
with open(f"preprocessed/{TASK}/testDataset.pkl", "wb") as f:
    pickle.dump(test, f)